# Lab 6.2 - LLM-Powered Log Summariser
**Module 6: Building AI-Powered Operations Tools**

In this lab you will:
- **Ingest** unstructured Nutanix system error logs from multiple CVM components
- **Design** a structured JSON output schema for machine-readable incident summaries
- **Batch-process** multiple incident types (disk, network, memory) in a single pipeline
- **Generate** a formatted incident report with root-cause analysis and remediation steps
- **Compare** LLM-based summarisation against the rule-based regex parser from Module 1
- **Pre-filter** logs with regex before sending to the LLM to reduce token cost

> **Instructor Note:** This is the **main hands-on project** for Module 6. The `NutanixLogSummariser` class built here becomes the core business logic for a FastAPI microservice in Module 7. Budget roughly **40 minutes** for this lab — 20 min building the summariser, 10 min batch processing, 10 min the comparison section. Learners who finish early should attempt the Challenges at the bottom.

## Requirements & Troubleshooting

### Required Packages

| Package | Install Name | Notes |
|---------|-------------|-------|
| google-generativeai | `google-generativeai` | Gemini API client |
| pandas | `pandas` | Results DataFrame display |
| pathlib | stdlib | File path handling |
| json | stdlib | JSON parsing |
| datetime | stdlib | Timestamp generation |
| re | stdlib | Regex pre-filtering |

**Install non-stdlib packages:**
```bash
pip install google-generativeai pandas
```

### API Key Required

This lab calls the **Google Gemini API**. Set your key before running:

```bash
export GEMINI_API_KEY="AIza..."
```

Get a free key at: [aistudio.google.com](https://aistudio.google.com/app/apikey)

### Synthetic Logs

> **Note:** All log data in this lab is **synthetically generated** by the `LogGenerator` class in Section 2. No external log files are needed. The patterns are based on real Nutanix AOS component signatures but do not represent any real cluster or customer data.

---

### Common Errors & Fixes

**`ModuleNotFoundError: No module named 'google.generativeai'`**
> Run `pip install google-generativeai` in a terminal, then **restart the kernel**.

**`EnvironmentError: GEMINI_API_KEY not set`**
> Export the key in your shell before launching Jupyter: `export GEMINI_API_KEY="AIza..."`

**`JSONDecodeError` when parsing LLM response**
> The LLM occasionally wraps JSON in markdown fences. The `analyse()` method strips these automatically and retries. If it persists, check the raw response printed in the error.

**`ResourceExhausted` / 429 rate limit**
> The free Gemini tier has per-minute limits. The `call_gemini()` helper adds a 1-second delay between calls. If batch processing hits rate limits, increase `time.sleep()` in the helper.

**`CalledProcessError` / `--break-system-packages`**
> You are in a venv. Run `pip install google-generativeai pandas` directly in a terminal with the venv activated.

In [3]:
# Auto-install required packages
import sys
import subprocess

REQUIRED = ["google-generativeai", "pandas"]

for pkg in REQUIRED:
    try:
        import_name = pkg.replace("-", "_").replace("google_generativeai", "google.generativeai")
        __import__(import_name.split(".")[0])
        print(f"  {pkg} already installed")
    except ImportError:
        print(f"  Installing {pkg}...")
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", pkg, "-q"],
            capture_output=True, text=True
        )
        if result.returncode != 0:
            print(f"    ERROR: {result.stderr[:200]}")
        else:
            print(f"    OK")

print("\nAll packages ready.")

  google-generativeai already installed
  pandas already installed

All packages ready.


In [4]:
import os
import json
import re
import time
from pathlib import Path
from datetime import datetime, timedelta

import google.generativeai as genai
import pandas as pd

# ── API configuration ──────────────────────────────────────────────────────
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
if not GEMINI_API_KEY:
    raise EnvironmentError(
        "GEMINI_API_KEY not set. "
        "Export it before launching Jupyter: export GEMINI_API_KEY='AIza...'"
    )

MODEL = "gemini-3.1-flash-lite"   # Flash: fast + cheap, ideal for log analysis

# ── Call log: records every API call for cost tracking ────────────────────
CALL_LOG: list[dict] = []

# Gemini 1.5 Flash pricing (USD per million tokens, as of 2025)
_PRICE_INPUT_PER_M  = 0.075   # $0.075 / 1M input tokens
_PRICE_OUTPUT_PER_M = 0.30    # $0.30  / 1M output tokens


def call_gemini(
    model_obj,
    prompt: str,
    label: str = "unlabelled",
    delay: float = 1.0,
) -> genai.types.GenerateContentResponse:
    """Call Gemini, log the call, and return the response.

    Args:
        model_obj: Configured GenerativeModel instance.
        prompt:    Full prompt string.
        label:     Human-readable label for cost tracking.
        delay:     Seconds to sleep after call (rate-limit buffer).

    Returns:
        GenerateContentResponse from the Gemini SDK.
    """
    start = time.time()
    response = model_obj.generate_content(prompt)
    elapsed = time.time() - start

    # Extract token counts if available
    try:
        in_tokens  = response.usage_metadata.prompt_token_count
        out_tokens = response.usage_metadata.candidates_token_count
    except Exception:
        in_tokens, out_tokens = 0, 0

    cost_usd = (
        (in_tokens  / 1_000_000) * _PRICE_INPUT_PER_M +
        (out_tokens / 1_000_000) * _PRICE_OUTPUT_PER_M
    )

    CALL_LOG.append({
        "label":       label,
        "timestamp":   datetime.now().isoformat(timespec="seconds"),
        "in_tokens":   in_tokens,
        "out_tokens":  out_tokens,
        "cost_usd":    round(cost_usd, 6),
        "latency_s":   round(elapsed, 2),
    })

    if delay > 0:
        time.sleep(delay)

    return response


# Quick connectivity check
genai.configure(api_key=GEMINI_API_KEY)
_probe = genai.GenerativeModel(MODEL)
_r = call_gemini(_probe, "Reply with the single word: READY", label="probe", delay=0)
print("API status:", _r.text.strip())
print(f"Model     : {MODEL}")
print(f"Pricing   : ${_PRICE_INPUT_PER_M}/M input  ${_PRICE_OUTPUT_PER_M}/M output")

API status: READY
Model     : gemini-3.1-flash-lite
Pricing   : $0.075/M input  $0.3/M output


## Section 1 - The Problem: Mean Time to Insight

When a Nutanix CVM starts failing, dozens of components log errors simultaneously. An on-call engineer receiving a PagerDuty alert at 2 AM faces a log stream like this:

```
2025-05-29 02:14:33.001 disk_manager disk_manager.cc:441] WARNING SMART attribute 197 (Current Pending Sector) value=12 threshold=0 on /dev/sdb node=NTNX-CVM-03
2025-05-29 02:14:35.112 disk_manager disk_manager.cc:502] ERROR Read error on /dev/sdb sector 0x1A4F: I/O error (errno=5)
2025-05-29 02:14:36.009 disk_manager disk_manager.cc:614] ERROR Disk /dev/sdb marked BAD after 3 consecutive I/O failures — removing from storage pool
2025-05-29 02:14:36.441 stargate stargate.cc:1203] ERROR Disk removal event received for vdisk_id=8812 — triggering data migration
2025-05-29 02:14:37.090 stargate stargate.cc:1889] FATAL Storage pool degraded: RF2 protection violated for container default-container-123
2025-05-29 02:14:37.201 stargate stargate.cc:2001] ERROR Stargate crash loop detected — restarting (attempt 1/3)
2025-05-29 02:14:39.774 cerebro cerebro.cc:334] WARNING Replication lag on ECG 44 has exceeded 120s — 3 VMs affected
2025-05-29 02:14:41.002 cerebro cerebro.cc:512] ERROR Protection domain pd-prod-vms unable to complete snapshot — Stargate unavailable
2025-05-29 02:14:42.555 cluster_health cluster_health.cc:88] ERROR Cluster health degraded: 1 node reporting storage errors, data protection SLA at risk
2025-05-29 02:14:43.100 genesis genesis.cc:201] WARNING Service stargate on NTNX-CVM-03 has restarted 2 times in last 60s
2025-05-29 02:14:44.800 cassandra cassandra.cc:77] WARNING Ring disruption detected — metadata writes may be delayed
2025-05-29 02:14:46.007 cluster_health cluster_health.cc:210] FATAL Cluster entering read-only mode to prevent data corruption
```

**The MTTI (Mean Time to Insight) problem:** A skilled L2 engineer needs 5-15 minutes to:
1. Identify the **first triggering event** (SMART sector errors on `/dev/sdb`) vs downstream symptoms (Cerebro lag, Cassandra ring disruption)
2. Determine **severity** (P1 — cluster entering read-only mode)
3. Recall the **correct remediation** (`ncli disk remove`, `ncc health_checks run_all`)
4. Decide whether to **escalate** to Nutanix Support

An LLM does this in under 3 seconds and formats the result as structured JSON that downstream systems (PagerDuty, Jira, Slack) can consume directly.

> **Instructor Note:** Walk learners through the **causal chain** in the log above: SMART warnings (precursor) → I/O errors (disk failing) → disk marked bad (trigger) → Stargate crash loop (first-order consequence) → Cerebro replication lag (second-order) → cluster read-only (final state). This chain is exactly what the LLM will be asked to extract. The rule-based parser from Module 1 can detect the SMART pattern but cannot reason about the causal chain — that is the key differentiator.

## Section 2 - Synthetic Log Generator

To keep this lab self-contained, we generate realistic log sequences programmatically. Each incident type follows the authentic Nutanix AOS log format:

```
YYYY-MM-DD HH:MM:SS.mmm  component  file.cc:line]  LEVEL  message
```

Three incident types cover the most common on-call scenarios for Nutanix engineers.

In [5]:
class LogGenerator:
    """Generate synthetic Nutanix AOS log sequences for common incident types."""

    _FMT = "%Y-%m-%d %H:%M:%S.%f"

    @staticmethod
    def _ts(base: datetime, delta_s: float) -> str:
        """Return a formatted timestamp offset by delta_s seconds from base."""
        t = base + timedelta(seconds=delta_s)
        return t.strftime("%Y-%m-%d %H:%M:%S") + f".{t.microsecond // 1000:03d}"

    def generate_disk_failure_incident(self, node: str = "NTNX-CVM-03", disk: str = "/dev/sdb") -> list[str]:
        """18-line disk SMART failure cascade: SMART → I/O → disk bad → Stargate crash → Cerebro lag → cluster degraded."""
        b = datetime(2025, 5, 29, 2, 14, 33)
        d = disk
        n = node
        return [
            f"{self._ts(b, 0.001)} disk_manager disk_manager.cc:441] WARNING SMART attribute 197 (Current Pending Sector) value=12 threshold=0 on {d} node={n}",
            f"{self._ts(b, 0.890)} disk_manager disk_manager.cc:441] WARNING SMART attribute 5 (Reallocated Sector Count) value=8 threshold=0 on {d} node={n}",
            f"{self._ts(b, 2.112)} disk_manager disk_manager.cc:502] ERROR Read error on {d} sector 0x1A4F: I/O error (errno=5)",
            f"{self._ts(b, 2.990)} disk_manager disk_manager.cc:502] ERROR Read error on {d} sector 0x1A50: I/O error (errno=5)",
            f"{self._ts(b, 3.009)} disk_manager disk_manager.cc:614] ERROR Disk {d} marked BAD after 3 consecutive I/O failures -- removing from storage pool node={n}",
            f"{self._ts(b, 3.204)} hades hades.cc:98]  INFO  Disk removal request queued for serial SN-4482AC disk={d}",
            f"{self._ts(b, 3.441)} stargate stargate.cc:1203] ERROR Disk removal event received for vdisk_id=8812 -- triggering data migration",
            f"{self._ts(b, 4.555)} stargate stargate.cc:1401] WARNING RF2 data migration in progress: 14 extents pending, ETA 240s",
            f"{self._ts(b, 4.889)} stargate stargate.cc:1889] FATAL Storage pool degraded: RF2 protection violated for container default-container-123",
            f"{self._ts(b, 5.001)} stargate stargate.cc:2001] ERROR Stargate crash loop detected -- restarting (attempt 1/3) node={n}",
            f"{self._ts(b, 6.774)} cerebro cerebro.cc:334] WARNING Replication lag on ECG 44 has exceeded 120s -- 3 VMs affected",
            f"{self._ts(b, 7.002)} cerebro cerebro.cc:512] ERROR Protection domain pd-prod-vms unable to complete snapshot -- Stargate unavailable",
            f"{self._ts(b, 8.002)} stargate stargate.cc:2001] ERROR Stargate crash loop detected -- restarting (attempt 2/3) node={n}",
            f"{self._ts(b, 8.555)} cluster_health cluster_health.cc:88] ERROR Cluster health degraded: 1 node reporting storage errors, data protection SLA at risk",
            f"{self._ts(b, 9.100)} genesis genesis.cc:201] WARNING Service stargate on {n} has restarted 2 times in last 60s",
            f"{self._ts(b, 10.800)} cassandra cassandra.cc:77] WARNING Ring disruption detected -- metadata writes may be delayed",
            f"{self._ts(b, 11.007)} cluster_health cluster_health.cc:210] FATAL Cluster entering read-only mode to prevent data corruption",
            f"{self._ts(b, 11.500)} genesis genesis.cc:315] ERROR Service stargate on {n} failed to restart after 3 attempts -- marking node degraded",
        ]

    def generate_network_partition_incident(self, node: str = "NTNX-CVM-07") -> list[str]:
        """15-line network partition: OVS bridge warnings → LACP flap → Zookeeper quorum loss → cluster unavailable."""
        b = datetime(2025, 5, 29, 3, 45, 10)
        n = node
        return [
            f"{self._ts(b, 0.100)} ovs ovs-vswitchd.cc:88] WARNING OVS bridge br0 packet drop rate elevated: 4.2% last 30s node={n}",
            f"{self._ts(b, 0.880)} ovs ovs-vswitchd.cc:120] ERROR LACP PDU timeout on bond0 -- partner system not responding node={n}",
            f"{self._ts(b, 1.450)} ovs ovs-vswitchd.cc:135] ERROR Bond bond0 member eth0 marked down -- failover to eth1 node={n}",
            f"{self._ts(b, 2.001)} ovs ovs-vswitchd.cc:140] WARNING LACP flap detected on bond0: 3 state changes in 5s node={n}",
            f"{self._ts(b, 2.750)} zookeeper zookeeper.cc:201] WARNING Lost connection to ZooKeeper quorum member 192.168.5.3",
            f"{self._ts(b, 3.100)} zookeeper zookeeper.cc:201] WARNING Lost connection to ZooKeeper quorum member 192.168.5.4",
            f"{self._ts(b, 3.450)} zookeeper zookeeper.cc:305] ERROR ZooKeeper quorum lost -- only 1 of 3 members reachable from {n}",
            f"{self._ts(b, 4.002)} stargate stargate.cc:701] ERROR Lost ZooKeeper session -- pausing I/O operations node={n}",
            f"{self._ts(b, 4.500)} cerebro cerebro.cc:220] ERROR Cerebro leader election failed -- ZooKeeper unavailable",
            f"{self._ts(b, 5.100)} prism prism_gateway.cc:88] ERROR Prism Element cluster view inconsistent -- node {n} unreachable",
            f"{self._ts(b, 5.800)} cluster_health cluster_health.cc:88] FATAL Network partition detected: {n} isolated from cluster",
            f"{self._ts(b, 6.200)} genesis genesis.cc:410] ERROR CVM services on {n} entering safe mode due to network isolation",
            f"{self._ts(b, 7.001)} acropolis acropolis.cc:55] ERROR VM migrations paused -- cluster network topology unstable",
            f"{self._ts(b, 8.500)} cluster_health cluster_health.cc:210] FATAL Cluster HA events suspended -- insufficient healthy nodes for quorum",
            f"{self._ts(b, 9.200)} genesis genesis.cc:512] WARNING Auto-recovery paused -- waiting for network partition resolution on {n}",
        ]

    def generate_memory_pressure_incident(self, node: str = "NTNX-CVM-05") -> list[str]:
        """12-line memory pressure cascade: CVM memory >90% → Cassandra OOM → Stargate degraded → VMs impacted."""
        b = datetime(2025, 5, 29, 11, 22, 5)
        n = node
        return [
            f"{self._ts(b, 0.050)} cvm_monitor cvm_monitor.cc:44] WARNING CVM memory utilisation at 91.3% on {n} (threshold: 90%)",
            f"{self._ts(b, 0.600)} cvm_monitor cvm_monitor.cc:44] WARNING CVM memory utilisation at 94.7% on {n} -- consider migrating VMs",
            f"{self._ts(b, 1.200)} cassandra cassandra.cc:312] ERROR OutOfMemoryError in org.apache.cassandra.service.StorageService -- heap exhausted node={n}",
            f"{self._ts(b, 1.550)} cassandra cassandra.cc:330] FATAL Cassandra JVM killed by OOM killer -- restarting node={n}",
            f"{self._ts(b, 2.100)} stargate stargate.cc:880] ERROR Cassandra unavailable -- metadata reads failing for vdisk operations",
            f"{self._ts(b, 2.800)} stargate stargate.cc:901] WARNING Stargate operating in degraded mode -- metadata cache stale",
            f"{self._ts(b, 3.500)} acropolis acropolis.cc:190] WARNING VM I/O latency elevated: avg 280ms (baseline: 4ms) on {n}",
            f"{self._ts(b, 4.100)} acropolis acropolis.cc:205] ERROR 4 VMs reporting I/O timeouts on {n} -- guests may be unresponsive",
            f"{self._ts(b, 5.000)} cerebro cerebro.cc:440] WARNING Snapshot schedule delayed for pd-dev-vms -- Stargate metadata unavailable",
            f"{self._ts(b, 6.200)} cluster_health cluster_health.cc:88] ERROR Cluster health degraded: node {n} CVM services impaired",
            f"{self._ts(b, 7.800)} genesis genesis.cc:201] WARNING Service cassandra on {n} has restarted 1 time in last 120s",
            f"{self._ts(b, 9.100)} cvm_monitor cvm_monitor.cc:88] ERROR CVM memory at 97.1% on {n} -- emergency VM migration recommended",
        ]

    def generate_all(self) -> dict[str, list[str]]:
        """Return all three incident types as a named dictionary."""
        return {
            "disk_failure":       self.generate_disk_failure_incident(),
            "network_partition":  self.generate_network_partition_incident(),
            "memory_pressure":    self.generate_memory_pressure_incident(),
        }


# Instantiate and preview
log_gen  = LogGenerator()
incidents = log_gen.generate_all()

for name, lines in incidents.items():
    print(f"\n--- {name.upper()} ({len(lines)} lines) ---")
    for line in lines[:3]:
        print(" ", line)


--- DISK_FAILURE (18 lines) ---
  2025-05-29 02:14:33.001 disk_manager disk_manager.cc:441] WARNING SMART attribute 197 (Current Pending Sector) value=12 threshold=0 on /dev/sdb node=NTNX-CVM-03
  2025-05-29 02:14:33.890 disk_manager disk_manager.cc:441] WARNING SMART attribute 5 (Reallocated Sector Count) value=8 threshold=0 on /dev/sdb node=NTNX-CVM-03
  2025-05-29 02:14:35.112 disk_manager disk_manager.cc:502] ERROR Read error on /dev/sdb sector 0x1A4F: I/O error (errno=5)

--- NETWORK_PARTITION (15 lines) ---
  2025-05-29 03:45:10.100 ovs ovs-vswitchd.cc:88] WARNING OVS bridge br0 packet drop rate elevated: 4.2% last 30s node=NTNX-CVM-07
  2025-05-29 03:45:10.880 ovs ovs-vswitchd.cc:120] ERROR LACP PDU timeout on bond0 -- partner system not responding node=NTNX-CVM-07
  2025-05-29 03:45:11.450 ovs ovs-vswitchd.cc:135] ERROR Bond bond0 member eth0 marked down -- failover to eth1 node=NTNX-CVM-07

--- MEMORY_PRESSURE (12 lines) ---
  2025-05-29 11:22:05.050 cvm_monitor cvm_monitor.c

## Section 3 - Prompt Design & Output Schema

### Why Schema-First Prompt Design Matters

A plain prompt like *"summarise this log"* produces a narrative paragraph. Useful for reading, useless for automation. In a production AIOps pipeline, the LLM output must be:

- **Parseable** by downstream code (PagerDuty webhooks, Jira issue creation, Slack formatters)
- **Consistent** across incident types so a single consumer handles all cases
- **Actionable** — remediation steps with real CLI commands, not generic advice
- **Calibrated** — a confidence score so the system knows when to escalate to a human

We achieve this by including the full target schema in the system prompt and demanding raw JSON output with no commentary.

### Output JSON Schema

```json
{
  "incident_title": "<one-line human-readable summary>",
  "root_cause": "<first triggering event, not a downstream symptom>",
  "affected_component": "<primary Nutanix component: stargate | cerebro | cassandra | zookeeper | ovs | disk_manager | acropolis | genesis>",
  "severity": "<P1 | P2 | P3>",
  "confidence": <float 0.0-1.0>,
  "causal_chain": [
    "<step 1: root trigger>",
    "<step 2: first-order consequence>",
    "<step 3: second-order consequence>",
    "..."
  ],
  "remediation_steps": [
    "<step 1 with actual ncli/acli/ncc command where applicable>",
    "<step 2>",
    "..."
  ],
  "estimated_resolution_time": "<e.g. 15-30 minutes | 1-2 hours>",
  "requires_escalation": <true | false>,
  "kb_references": [
    "<KB article title or ID, e.g. KB-4567: Stargate crash loop after disk removal>"
  ]
}
```

### Severity Definitions

| Level | Criteria |
|-------|----------|
| **P1** | Data loss risk, cluster in read-only mode, or complete service unavailability |
| **P2** | Degraded redundancy (RF2 violated), multiple VMs impacted, no immediate data loss |
| **P3** | Single component warning, no user-facing impact yet |

> **Instructor Note:** The schema is a **product design decision**, not a technical one. Before writing the prompt, ask: *What does the consumer of this API need?* The `requires_escalation` boolean directly drives a PagerDuty escalation policy. The `confidence` float drives a human-review queue threshold. The `kb_references` array pre-populates a Jira ticket description. Every field should map to a downstream action — if it doesn't, remove it. This is a key lesson in LLM system design.

## Section 4 - NutanixLogSummariser Class

In [11]:
class NutanixLogSummariser:
    """LLM-powered summariser for Nutanix AOS component logs.

    Sends raw log lines to Gemini and returns a structured JSON dict
    containing root-cause analysis, causal chain, severity, and
    remediation steps with real Nutanix CLI commands.
    """

    SYSTEM_PROMPT = """You are a senior Nutanix L2 Support Engineer with 8+ years of experience \
diagnosing AOS component failures in production clusters. You have deep expertise in Stargate, \
Cerebro, Cassandra, Zookeeper, Genesis, Acropolis, and disk management subsystems.

Rules:
1. Identify the FIRST TRIGGERING EVENT in the log, not downstream symptoms. \
   A Stargate crash caused by a disk removal is a disk failure incident, not a Stargate incident.
2. Severity definitions: \
   P1=data loss risk or cluster entering read-only/unavailable state; \
   P2=degraded redundancy or multiple VMs impacted without immediate data loss; \
   P3=single component warning without user-facing impact.
3. Remediation steps MUST use real Nutanix CLI commands where applicable: \
   ncli, acli, ncc, nutanix_guest_tools_cli, or standard Linux commands on the CVM.
4. Return ONLY a raw JSON object. No markdown, no code fences, no prose before or after the JSON.
5. If confidence is below 0.6, set requires_escalation to true regardless of severity."""

    _SCHEMA_HINT = """Return a JSON object with these exact keys:
incident_title, root_cause, affected_component, severity (P1/P2/P3),
confidence (0.0-1.0), causal_chain (list of strings), remediation_steps (list of strings),
estimated_resolution_time, requires_escalation (bool), kb_references (list of strings)."""

    def __init__(self, model: str = MODEL, api_key: str = GEMINI_API_KEY):
        genai.configure(api_key=api_key)
        self._model = genai.GenerativeModel(
            model,
            system_instruction=self.SYSTEM_PROMPT,
        )
        self.model_name = model

    def _build_prompt(self, log_lines: list[str]) -> str:
        """Format the user prompt with schema hint and log lines."""
        log_block = "\n".join(log_lines)
        return (
            f"{self._SCHEMA_HINT}\n\n"
            f"Analyse {len(log_lines)} log lines from a Nutanix cluster and return the JSON object:\n\n"
            f"{log_block}"
        )

    def analyse(
        self,
        log_lines: list[str],
        incident_name: str = "incident",
        max_retries: int = 2,
    ) -> dict:
        """Send log lines to Gemini and return parsed JSON dict.

        Strips markdown fences and retries up to max_retries times on
        JSONDecodeError before raising.
        """
        prompt = self._build_prompt(log_lines)

        for attempt in range(1, max_retries + 2):
            response = call_gemini(self._model, prompt, label=incident_name)
            raw_text = response.text.strip()

            # Strip markdown code fences if present
            cleaned = re.sub(r"^```(?:json)?\s*", "", raw_text, flags=re.IGNORECASE)
            cleaned = re.sub(r"\s*```$", "", cleaned).strip()

            try:
                result = json.loads(cleaned)
                result["_incident_name"] = incident_name
                result["_log_line_count"] = len(log_lines)
                return result
            except json.JSONDecodeError as exc:
                if attempt <= max_retries:
                    print(f"  [retry {attempt}/{max_retries}] JSONDecodeError for '{incident_name}': {exc}")
                    time.sleep(1)
                else:
                    print(f"  [FAILED] Could not parse JSON for '{incident_name}' after {max_retries} retries")
                    print(f"  Raw response (first 400 chars): {raw_text[:400]}")
                    raise

    def analyse_batch(
        self,
        incidents_dict: dict[str, list[str]],
    ) -> list[dict]:
        """Analyse multiple incidents sequentially. Returns list of result dicts."""
        results = []
        total = len(incidents_dict)
        for i, (name, lines) in enumerate(incidents_dict.items(), start=1):
            print(f"[{i}/{total}] Analysing '{name}' ({len(lines)} lines)...")
            result = self.analyse(lines, incident_name=name)
            result["incident_name"] = name  # top-level key for easy access
            results.append(result)
            print(f"       -> {result.get('severity', '?')} | {result.get('incident_title', '?')[:60]}")
        return results

    def generate_report(self, results: list[dict]) -> str:
        """Format a list of analysis results as a Markdown incident report."""
        now = datetime.now().strftime("%Y-%m-%d %H:%M:%S UTC")
        lines = [
            "# Nutanix Cluster Incident Report",
            f"*Generated: {now}*",
            f"*Incidents analysed: {len(results)}*",
            "",
            "---",
        ]
        for r in results:
            sev   = r.get("severity", "?")
            title = r.get("incident_title", r.get("incident_name", "Unknown"))
            conf  = r.get("confidence", 0)
            esc   = "YES - escalate to Nutanix Support" if r.get("requires_escalation") else "No"
            lines += [
                f"",
                f"## [{sev}] {title}",
                f"**Incident ID:** `{r.get('incident_name', 'unknown')}`  ",
                f"**Confidence:** {conf:.0%}  ",
                f"**Escalation:** {esc}  ",
                f"**ETA:** {r.get('estimated_resolution_time', 'Unknown')}  ",
                "",
                f"### Root Cause",
                r.get("root_cause", ""),
                "",
                f"### Causal Chain",
            ]
            for step_i, step in enumerate(r.get("causal_chain", []), start=1):
                lines.append(f"{step_i}. {step}")
            lines += ["", "### Remediation Steps"]
            for step_i, step in enumerate(r.get("remediation_steps", []), start=1):
                lines.append(f"{step_i}. {step}")
            kb = r.get("kb_references", [])
            if kb:
                lines += ["", "### KB References"]
                for ref in kb:
                    lines.append(f"- {ref}")
            lines += ["", "---"]
        return "\n".join(lines)

    def get_cost_summary(self) -> dict:
        """Return aggregated token and cost summary from CALL_LOG."""
        relevant = [c for c in CALL_LOG if c["label"] != "probe"]
        total_in  = sum(c["in_tokens"]  for c in relevant)
        total_out = sum(c["out_tokens"] for c in relevant)
        total_cost = sum(c["cost_usd"]  for c in relevant)
        return {
            "total_calls":      len(relevant),
            "total_in_tokens":  total_in,
            "total_out_tokens": total_out,
            "total_cost_usd":   round(total_cost, 6),
            "avg_latency_s":    round(
                sum(c["latency_s"] for c in relevant) / max(len(relevant), 1), 2
            ),
        }


# ── Quick smoke test on one incident ──────────────────────────────────────
print("Instantiating NutanixLogSummariser...")
summariser = NutanixLogSummariser()

print(f"Testing on disk_failure incident ({len(incidents['disk_failure'])} lines)...")
single_result = summariser.analyse(incidents["disk_failure"], incident_name="disk_failure")

print("\nResult:")
print(json.dumps(single_result, indent=2))

Instantiating NutanixLogSummariser...
Testing on disk_failure incident (18 lines)...

Result:
{
  "incident_title": "Disk Hardware Failure leading to Stargate Crash and Read-Only Cluster State",
  "root_cause": "Physical disk failure (SSD/HDD) on /dev/sdb on node NTNX-CVM-03 causing I/O errors, subsequent Stargate crash loop, and loss of RF2 redundancy triggering cluster-wide read-only mode.",
  "affected_component": "Disk /dev/sdb (Storage Subsystem)",
  "severity": "P1",
  "confidence": 0.95,
  "causal_chain": [
    "SMART attribute degradation (Current Pending/Reallocated Sectors) on /dev/sdb",
    "Physical I/O errors (errno 5) triggered by sector read failure",
    "Disk /dev/sdb marked BAD and removed from storage pool",
    "RF2 data migration triggered by disk removal",
    "Stargate crash loop due to inability to sustain RF2 writes",
    "Cluster enters read-only mode to prevent data corruption"
  ],
  "remediation_steps": [
    "Run 'ncc health_checks hardware_checks disk_che

## Section 5 - Batch Processing

### Processing Multiple Incidents

In a real AIOps pipeline, incidents arrive in parallel from multiple CVMs. The `analyse_batch()` method processes them sequentially with rate-limit protection. For production, you would use `asyncio` or a thread pool — covered in Module 8.

### Cost Model

Each log analysis call consumes roughly:
- **Input:** 300-500 tokens (system prompt ~180 tokens + 15-20 log lines ~150-250 tokens)
- **Output:** 250-400 tokens (structured JSON response)
- **Cost per incident:** ~$0.00015-0.00025 USD with Gemini 1.5 Flash
- **Cost for 1,000 incidents/day:** ~$0.15-0.25 USD

> **Instructor Note:** The **economics of LLM-powered operations** are compelling at this scale. $0.20/day to automatically triage 1,000 incidents vs. a L2 engineer spending 8 minutes on each one. The key design lever is **pre-filtering** (Section 7) — reducing tokens sent to the LLM directly reduces cost. A well-designed regex pre-filter can cut input tokens by 40-60%, halving the daily cost while maintaining accuracy.

In [7]:
print("Running batch analysis on all 3 incidents...\n")
all_results = summariser.analyse_batch(incidents)

# Display as a summary DataFrame
rows = []
for r in all_results:
    rows.append({
        "Incident":  r.get("incident_name", "?"),
        "Title":     r.get("incident_title", "")[:55],
        "Severity":  r.get("severity", "?"),
        "Component": r.get("affected_component", "?"),
        "Confidence": f"{r.get('confidence', 0):.0%}",
        "Escalate":  "YES" if r.get("requires_escalation") else "no",
        "ETA":       r.get("estimated_resolution_time", "?"),
    })

df = pd.DataFrame(rows)
print("\nBatch Results:")
print(df.to_string(index=False))

Running batch analysis on all 3 incidents...

[1/3] Analysing 'disk_failure' (18 lines)...
       -> P1 | Disk Hardware Failure leading to Stargate crash and cluster-
[2/3] Analysing 'network_partition' (15 lines)...
       -> P1 | Network Partition via LACP/Physical Switch Link Failure on N
[3/3] Analysing 'memory_pressure' (12 lines)...
       -> P2 | Cassandra OutOfMemory (OOM) due to CVM Memory Exhaustion

Batch Results:
         Incident                                                   Title Severity                         Component Confidence Escalate            ETA
     disk_failure Disk Hardware Failure leading to Stargate crash and clu       P1 Storage Subsystem (Disk /dev/sdb)        95%       no      4-6 hours
network_partition Network Partition via LACP/Physical Switch Link Failure       P1 Network/Bonding Interface (bond0)        95%       no 60-120 minutes
  memory_pressure Cassandra OutOfMemory (OOM) due to CVM Memory Exhaustio       P2                         Cassandr

## Section 6 - Incident Report Generation

### Downstream Uses of the Report

The `generate_report()` method produces Markdown that feeds multiple downstream systems:

- **PagerDuty** — incident title + severity map to alert priority; `requires_escalation` triggers an on-call page
- **Jira** — `remediation_steps` + `kb_references` pre-populate a ticket description, cutting ticket creation time from 5 min to 30 sec
- **Slack** — the `root_cause` + `causal_chain` fields render in a bot message for the #ops-alerts channel
- **Module 7 FastAPI** — in the next module, `generate_report()` becomes the response body of a `POST /analyse` endpoint that Prism can call directly

In [8]:
# Generate the formatted incident report
report_md = summariser.generate_report(all_results)

print(report_md)

# Save to file for Module 7 FastAPI integration
report_path = Path("incident_report.md")
report_path.write_text(report_md, encoding="utf-8")
print(f"\nReport saved to: {report_path.resolve()}")

# Cost summary
cost = summariser.get_cost_summary()
print("\n--- API Cost Summary ---")
for k, v in cost.items():
    print(f"  {k:<22} {v}")

# Nutanix Cluster Incident Report
*Generated: 2026-05-29 13:44:21 UTC*
*Incidents analysed: 3*

---

## [P1] Disk Hardware Failure leading to Stargate crash and cluster-wide I/O suspension
**Incident ID:** `disk_failure`  
**Confidence:** 95%  
**Escalation:** No  
**ETA:** 4-6 hours  

### Root Cause
Physical disk failure (/dev/sdb) on NTNX-CVM-03 characterized by increasing Reallocated and Pending Sectors followed by persistent I/O errors (errno=5), causing Stargate crash loops and metadata inconsistency.

### Causal Chain
1. SMART attribute 197/5 warnings on /dev/sdb
2. I/O error (errno=5) read failure
3. Disk /dev/sdb marked BAD by disk_manager
4. Stargate triggered data migration
5. RF2 protection violation detected
6. Stargate service crash loop
7. Cluster enters read-only mode to prevent data corruption

### Remediation Steps
1. Run 'ncc health_checks hardware_checks disk_check' to verify disk status across the cluster.
2. Identify the physical slot for the failed disk using 'nc

## Section 7 - LLM vs Rule-Based Analysis: A Comparison

### Capability Comparison

This course has built three generations of log analysis tools. Here is how they compare:

| Capability | Module 1: Regex Parser | Module 4: NLP Pipeline | Lab 6.2: LLM Summariser |
|---|---|---|---|
| Parse log format | Yes (fixed patterns) | Yes (tokenisation) | Yes (any format) |
| Extract root cause | No (pattern match only) | Partial (entity recognition) | Yes (causal reasoning) |
| Causal chain reconstruction | No | No | Yes |
| Remediation steps (CLI commands) | No | No | Yes |
| Novel / unseen patterns | No (breaks) | Partial | Yes |
| Handles log format variations | No | Partial | Yes |
| Cost per 1,000 incidents | ~$0 (CPU only) | ~$0 (CPU only) | ~$0.20 |
| Latency per incident | <1 ms | 10-50 ms | 1-3 s |
| Auditability (deterministic) | Yes | Yes | Partial (non-deterministic) |
| Structured output | Manual (code) | Manual (code) | Schema-prompted |

### Pre-Filtering: Getting the Best of Both Worlds

The cost and latency disadvantages of LLMs can be significantly reduced by using regex to **pre-filter** logs before sending them to the API. Only ERROR and FATAL lines carry causal information; DEBUG and INFO lines add tokens without adding insight.

> **Instructor Note:** This is the core **pipeline architecture** lesson. In production, you compose all three approaches: regex pre-filter (Module 1) strips noise, NLP entity extractor (Module 4) tags components, LLM (Module 6) reasons over the filtered+tagged lines. Each layer adds value at its appropriate cost point. The filter below typically reduces token usage by 40-60% while preserving the full causal chain — because every link in the chain is an ERROR or FATAL event.

In [9]:
def prefilter_logs(lines: list[str], min_severity: str = "ERROR") -> list[str]:
    """Filter log lines to only include entries at or above min_severity.

    Severity order: DEBUG < INFO < WARNING < ERROR < FATAL

    Args:
        lines:        Raw log lines.
        min_severity: Minimum severity level to retain (case-insensitive).

    Returns:
        Filtered list containing only lines at or above min_severity.
    """
    _ORDER = {"DEBUG": 0, "INFO": 1, "WARNING": 2, "ERROR": 3, "FATAL": 4}
    threshold = _ORDER.get(min_severity.upper(), 3)

    # Pattern: matches severity token after the component/file prefix
    _SEV_RE = re.compile(
        r"\]\s+(DEBUG|INFO|WARNING|ERROR|FATAL)\s",
        re.IGNORECASE,
    )

    filtered = []
    for line in lines:
        m = _SEV_RE.search(line)
        if m:
            level = m.group(1).upper()
            if _ORDER.get(level, 0) >= threshold:
                filtered.append(line)
        # Lines without a parseable severity are included (conservative)
    return filtered


# ── Show token reduction for each incident ────────────────────────────────
print("Token reduction analysis (approximate, ~4 chars per token):\n")
print(f"{'Incident':<22} {'Original':>8} {'Filtered':>8} {'Reduction':>10} {'Lines kept':>10}")
print("-" * 65)

filtered_incidents = {}
for name, lines in incidents.items():
    filtered = prefilter_logs(lines, min_severity="ERROR")
    filtered_incidents[name] = filtered

    orig_chars = sum(len(l) for l in lines)
    filt_chars = sum(len(l) for l in filtered)
    orig_tok   = orig_chars // 4
    filt_tok   = filt_chars // 4
    reduction  = (1 - filt_tok / max(orig_tok, 1)) * 100

    print(
        f"{name:<22} {orig_tok:>8,} {filt_tok:>8,} "
        f"{reduction:>9.1f}% {len(filtered):>5}/{len(lines)}"
    )

# ── Re-analyse disk_failure with filtered logs ────────────────────────────
print("\nRe-analysing disk_failure with ERROR/FATAL-only logs...")
filtered_disk_result = summariser.analyse(
    filtered_incidents["disk_failure"],
    incident_name="disk_failure_filtered",
)

print("\nComparison: Full logs vs Filtered logs")
print(f"  Full logs    - severity: {single_result.get('severity')}  confidence: {single_result.get('confidence'):.2f}  root_cause: {single_result.get('root_cause', '')[:60]}")
print(f"  Filtered logs - severity: {filtered_disk_result.get('severity')}  confidence: {filtered_disk_result.get('confidence'):.2f}  root_cause: {filtered_disk_result.get('root_cause', '')[:60]}")
print()
print("Observation: Filtering typically preserves severity and root cause while reducing token cost.")

Token reduction analysis (approximate, ~4 chars per token):

Incident               Original Filtered  Reduction Lines kept
-----------------------------------------------------------------
disk_failure                600      374      37.7%    11/18
network_partition           465      311      33.1%    10/15
memory_pressure             387      199      48.6%     6/12

Re-analysing disk_failure with ERROR/FATAL-only logs...

Comparison: Full logs vs Filtered logs
  Full logs    - severity: P1  confidence: 0.95  root_cause: Physical disk failure (SMART attribute 197/5) on /dev/sdb ca
  Filtered logs - severity: P1  confidence: 0.95  root_cause: Physical disk failure (/dev/sdb) on node NTNX-CVM-03 causing

Observation: Filtering typically preserves severity and root cause while reducing token cost.


## Summary

In this lab you built a complete LLM-powered log analysis pipeline from scratch.

### Components Built

| Component | Purpose | Key Design Decision |
|-----------|---------|--------------------|
| `LogGenerator` | Synthetic Nutanix AOS log sequences | Realistic causal chains across 3 incident types |
| `call_gemini()` helper | API wrapper with cost tracking | Logs every call to `CALL_LOG` for economics analysis |
| `NutanixLogSummariser.SYSTEM_PROMPT` | L2 engineer persona + rules | Identifies root trigger, not symptoms; forces raw JSON |
| `_build_prompt()` | Schema hint + log block | Schema embedded in prompt ensures consistent output |
| `analyse()` | Single incident analysis | JSON fence stripping + retry on `JSONDecodeError` |
| `analyse_batch()` | Multi-incident pipeline | Sequential with rate-limit protection |
| `generate_report()` | Markdown incident report | Structured for PagerDuty / Jira / Slack consumption |
| `prefilter_logs()` | Regex pre-filter | 40-60% token reduction while preserving causal chain |

### What Comes Next

- **Lab 6.3** — Connects the summariser to Jira (create incident tickets) and Slack (post alert summaries) via MCP tool use
- **Module 7** — Wraps `NutanixLogSummariser` as a FastAPI microservice with a `POST /analyse` endpoint, authentication, and a simple Prism-compatible webhook receiver

## Challenges

### Challenge 1 - Confidence Calibration

The LLM's `confidence` score is self-reported and may not be well-calibrated. Design an experiment to test it:

1. Create three modified versions of the `disk_failure` incident: (a) full logs, (b) only the first 5 lines (incomplete), (c) logs with misleading symptoms added (e.g., insert unrelated Cassandra warnings before the SMART lines)
2. Run `analyse()` on each and compare `confidence` values
3. Does the LLM correctly report lower confidence on the incomplete or misleading logs? If not, what does this imply for production use?

**Hint:** Consider adding a `_validate_confidence()` post-processing step that compares `confidence` against the ratio of ERROR/FATAL lines to total lines.

### Challenge 2 - Prompt Ablation

Test the impact of each rule in `SYSTEM_PROMPT` by removing them one at a time:

1. Remove Rule 1 (first triggering event). Does the LLM now report Stargate as root cause for the disk failure?
2. Remove Rule 3 (real CLI commands). What do the remediation steps look like without it?
3. Remove Rule 4 (raw JSON only). How often does the LLM add markdown fences or prose?

Document your findings. This is a real technique called **prompt ablation** — treating each instruction as a feature to be tested empirically.

### Challenge 3 - Streaming Analysis

The Gemini SDK supports streaming responses via `model.generate_content(..., stream=True)`. Streaming is valuable for long log files where the user needs to see partial results immediately.

1. Modify `call_gemini()` to accept a `stream=False` parameter
2. When `stream=True`, use the streaming API and print chunks as they arrive
3. Note: token counting requires waiting for the full response. Handle this by aggregating the streamed response and then parsing
4. Measure the **time-to-first-token** vs full latency. Is streaming worth it for logs of this size?

In [10]:
# Challenge workspace
# Write your challenge solutions here
